# Module 7: Testing Parallel Trends

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Everything so far rests on one assumption: that without the program, the two
groups would have changed by the same proportion.

It is a statement about a quantity nobody observes, so it cannot be verified.
What it can be is **tested against its own implication**, in the years before
the program existed, and this module does that three ways.

It also measures how much the test is worth, which is less than people assume.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

f["lo"] = np.log(f["n_arrests"])
pi = pd.PeriodIndex(f["year_month"], freq="M")
f["yr"] = pi.year.values + (pi.month.values - 1) / 12.0

pct = lambda b: 100 * (np.exp(b) - 1)
pre = f[f["period"] == "before"]          # the years before the program existed


def trend(sub):
    """Annual percentage change in the use of force rate, with an interval."""
    z = smf.glm("n_uof ~ yr", sub, family=sm.families.Poisson(),
                offset=sub["lo"]).fit()
    lo, hi = z.conf_int().loc["yr"]
    return pct(z.params["yr"]), pct(lo), pct(hi)


print(f"{pre['year_month'].nunique()} months before the program, "
      f"{f['year_month'].nunique()} in total")

## 2. Look at it first

Before any test, one plot of the pre period, one line per agency. Almost every
failed difference in differences would have been caught here.

In [ ]:
rows = []
for a in sorted(f["agency_id"].unique()):
    e, lo, hi = trend(pre[pre["agency_id"] == a])
    rows.append({"agency": NAME[a], "trained": "yes" if a in TRAINED else "",
                 "trend a year": f"{e:+.2f}%",
                 "95 percent interval": f"[{lo:+.2f}, {hi:+.2f}]"})
pd.DataFrame(rows).sort_values("trend a year").set_index("agency")

Two things are visible and the second is the one people miss.

**Summit County is an outlier.** It was falling at 12 percent a year, against
4 to 6 for everyone else, and its interval does not reach the others.

**Most of the intervals are enormous.** Orrindale's runs from 16 percent down
to 31 percent up. Only three of the twelve agencies have an interval that
excludes zero. **The per agency test barely works**, and it still catches
Summit County, which tells you how far out Summit County is.

## 3. The group level test

The formal version regresses the pre period on time, with an interaction
between time and the treated group. The interaction is the difference in
trends.

In [ ]:
for label, sub, trained_set in [
        ("Summit County excluded", pre[pre["agency_id"] != "A007"],
         [a for a in TRAINED if a != "A007"]),
        ("Summit County included", pre, TRAINED)]:
    s = sub.copy()
    s["tr"] = s["agency_id"].isin(trained_set).astype(float)
    z = smf.glm("n_uof ~ C(agency_id) + yr + tr:yr", s,
                family=sm.families.Poisson(), offset=s["lo"]).fit()
    k = [x for x in z.params.index if "yr" in x and "tr" in x][0]
    lo, hi = z.conf_int().loc[k]
    print(f"  {label:26s} difference in pre trends "
          f"{pct(z.params[k]):+5.2f}% a year  [{pct(lo):+5.2f}, {pct(hi):+5.2f}]"
          f"   p = {z.pvalues[k]:.4f}")

Read the second line carefully, because it is the most important output in
this module.

With Summit County in, the two groups' pre trends differ by 2.16 percent a
year and **p is 0.0875. That passes at the conventional threshold.** An
analyst who ran only this test would have concluded the design was sound, kept
Summit County, and reported 17.0 percent against a truth of 12.

The violation was obvious in the table in section 2 and nearly invisible to
the group test, because averaging Summit County with four well behaved
agencies dilutes it.

**A parallel trends test that passes is weak evidence.** It is mostly evidence
that the test has little power, which with a handful of agencies it does.

## 4. How much power does the test actually have

Rather than assert that, measure it. Plant a pre trend difference of a known
size and see how often the test finds it.

In [ ]:
rng = np.random.default_rng(4)
keep = [a for a in TRAINED if a != "A007"]
base = pre[pre["agency_id"].isin(keep + COMPARISON)].copy()

def detect_rate(extra_trend_pct, reps=200):
    """Share of simulated datasets where the test rejects at 0.05."""
    hits = 0
    for _ in range(reps):
        s = base.copy()
        s["tr"] = s["agency_id"].isin(keep).astype(float)
        mu = s["n_uof"] * np.exp(np.log(1 + extra_trend_pct / 100)
                                 * s["tr"] * (s["yr"] - s["yr"].min()))
        s["y"] = rng.poisson(np.maximum(mu, 0.01))
        z = smf.glm("y ~ C(agency_id) + yr + tr:yr", s,
                    family=sm.families.Poisson(), offset=s["lo"]).fit()
        k = [x for x in z.params.index if "yr" in x and "tr" in x][0]
        hits += z.pvalues[k] < 0.05
    return 100 * hits / reps

for extra in [1, 2, 3, 5]:
    print(f"  a planted difference of {extra} percent a year is found "
          f"{detect_rate(extra):3.0f} percent of the time")

The test finds a large violation reliably and a small one hardly at all.

That matters because **a small violation is enough to matter.** The group
level violation caused by Summit County is 2.16 percent a year, and a
violation that size is found **16 percent of the time**. The test missed it
here, and it would miss it in five cases out of six.

## 5. What the test can and cannot tell you

| If the test | It means | It does not mean |
|---|---|---|
| Fails | the design is in trouble, act | the program had no effect |
| Passes | nothing was detected | the trends are parallel |
| Passes with wide intervals | the test had no power | anything at all |

**Always report the interval on the pre trend difference, not just the p
value.** An interval from 4.6 percent down to 0.3 percent up, which is what
the A007 included case gives, says plainly that differences of 4 percent a
year have not been ruled out.

## Exercise

The test above uses a straight line through the pre period. Check whether the
two groups also match month by month, which is a stricter requirement.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    keep = [a for a in TRAINED if a != "A007"]
    s = pre[pre["agency_id"].isin(keep + COMPARISON)].copy()
    s["tr"] = s["agency_id"].isin(keep).astype(float)
    s["half"] = (s["year_month"] >= "2021-04").astype(float)      # split the pre period
    s["tr_half"] = s["tr"] * s["half"]
    z = smf.glm("n_uof ~ C(agency_id) + C(year_month) + tr_half", s,
                family=sm.families.Poisson(), offset=s["lo"]).fit()
    lo, hi = z.conf_int().loc["tr_half"]
    print("  a placebo difference in differences inside the pre period,")
    print("  splitting it in half at April 2021:\n")
    print(f"    estimate {pct(z.params['tr_half']):+.1f}%  "
          f"[{pct(lo):+.1f}, {pct(hi):+.1f}]")
    print(f"    the true effect of a program that had not started: {0.0:+.1f}%")
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

The placebo estimate is small and its interval comfortably contains zero,
which is what it should be: nothing happened in April 2021.

**This is a better test than the straight line version**, for two reasons. It
does not assume the pre trends are linear, and it uses exactly the estimator
that will be used for the real comparison, so if that estimator is going to
misbehave, it misbehaves here where the answer is known.

It is also a placebo test, which is [Module 12](Module_12_Placebo_Tests.ipynb),
and the fact that the parallel trends check and the placebo check turn out to
be the same procedure run at different dates is worth noticing rather than
memorising twice.

The one thing it shares with the straight line test is the weakness: a
placebo that fails to reject is not proof of anything. Report its interval.

</details>

---

**Next:** [Module 8: When Parallel Trends Fails](Module_08_When_Parallel_Trends_Fails.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*